In [1]:
import numpy as np
import scipy.optimize 

#### Ottimizzazione Numerica e L-BFGS

L'ottimizzazione numerica gioca un ruolo fondamentale nell'addestramento dei modelli di classificazione, come ad esempio la Regressione Logistica. A differenza dei modelli gaussiani, per i quali sono disponibili espressioni in forma chiusa per ottenere le soluzioni di Maximum Likelihood, la Regressione Logistica non ammette una soluzione analitica diretta. Il modello si ottiene infatti minimizzando la cross-entropia media calcolata tra le predizioni del modello stesso e le etichette effettivamente osservate, il che corrisponde alla ricerca della soluzione di Maximum Likelihood. Di conseguenza, per trovare il punto di minimo di questa funzione obiettivo, o in modo del tutto equivalente il massimizzatore delle verosimiglianze di classe, risulta strettamente necessario ricorrere all'utilizzo di algoritmi di ottimizzazione numerica.

#### Gradient Descent

Gli algoritmi di ottimizzazione numerica cercano i minimi di una data funzione $f(x)$ rispetto al suo argomento. Un metodo iterativo tra i più semplici per individuare un minimo locale della funzione $f$ è la discesa del gradiente, o gradient descent. Partendo da un punto iniziale $x_{t}$, questo algoritmo analizza l'andamento della funzione al fine di cercare una direzione di discesa. Tale direzione è data matematicamente dall'opposto del gradiente della funzione valutato in quel punto. A ogni singola iterazione, l'algoritmo calcola il punto successivo muovendosi lungo la direzione di discesa partendo da $x_{t}$ e compiendo un passo di dimensione $\alpha_{t}$. Questo passaggio cruciale è formalizzato dall'equazione di aggiornamento iterativo:

$$x_{t+1}=x_{t}-\alpha_{t}\nabla_{x}f(x)$$

Sotto alcune ipotesi non stringenti relative alla dimensione del passo $\alpha_{t}$, come ad esempio le condizioni per cui $\alpha_{t}\rightarrow0$ e simultaneamente $\sum_{t=1}^{\infty}\alpha_{t}\rightarrow\infty$, l'algoritmo garantisce la convergenza verso un minimo locale della funzione $f$. Uno degli svantaggi principali della discesa del gradiente risiede tuttavia nella sua lentezza. Per riuscire ad accelerare la convergenza verso il minimo, è possibile prendere in considerazione le informazioni del secondo ordine del dominio della funzione, introducendo il calcolo dell'Hessiana.

#### L-BFGS

A questo preciso scopo si inserisce l'algoritmo L-BFGS, un metodo avanzato che si basa sulla costruzione incrementale di un'approssimazione della matrice Hessiana, utilizzata per identificare una direzione di ricerca ottimale $p_{t}$ durante ogni iterazione. Una volta stabilita questa direzione di ricerca, l'algoritmo procede determinando una dimensione accettabile per il passo $\alpha_{t}$ per quella specifica direzione, impiegando poi congiuntamente direzione e dimensione del passo per aggiornare la soluzione corrente. L'implementazione pratica di questo ottimizzatore è disponibile nell'ecosistema Python tramite la libreria SciPy, interfacciandosi con il solutore numerico attraverso la funzione `scipy.optimize.fmin_l_bfgs_b`. Questa procedura richiede obbligatoriamente almeno due argomenti principali per essere eseguita: `func`, che rappresenta l'effettiva funzione obiettivo che si intende minimizzare, e `x0`, che definisce il valore o array di partenza per l'inizializzazione dell'algoritmo.

#### Calcolo del Gradiente

Per operare in maniera corretta, l'algoritmo L-BFGS richiede sia il calcolo della funzione obiettivo sia la conoscenza del suo gradiente. Lo strumento offre diverse alternative concettuali per passare il calcolo del gradiente all'interno della computazione. La prima modalità consiste nel costruire la funzione `func` affinché restituisca direttamente una tupla strutturata esattamente come $(f(x),\nabla_{x}f(x))$. Un approccio alternativo sfrutta il parametro opzionale `fprime`, mediante il quale si fornisce al sistema una seconda funzione interamente dedicata al calcolo matematico del gradiente; in questo scenario specifico, la funzione principale `func` dovrà limitarsi a restituire esclusivamente il valore obiettivo $f(x)$.

L'ultima opzione implementabile consiste nell'affidare completamente all'implementazione sottostante il calcolo di un gradiente approssimato, impostando l'argomento `approx_grad = True` durante la chiamata. Anche in quest'ultimo caso, la funzione `func` elaborerà unicamente il valore obiettivo $f(x)$. Questa comoda modalità operativa non richiede allo sviluppatore lo sforzo manuale di derivare formalmente la funzione e tradurla in codice, poiché l'approssimazione del gradiente viene calcolata in totale autonomia tramite il metodo numerico delle differenze finite. Ad esempio, il sistema valuta numericamente il gradiente calcolando la seguente equazione per le varie coordinate:

$$\frac{\partial f(x)}{\partial x_{i}}\approx\frac{f(x+\epsilon e_{i})-f(x-\epsilon e_{i})}{2\epsilon}$$

Nella definizione rigorosa di tale formula, $e_{i}$ rappresenta un vettore caratterizzato esclusivamente da zeri, fatta eccezione per l'elemento situato nella posizione $i$-esima che assume il valore unitario (ad esempio, per la prima coordinata si avrà $e_{1}=[1,0,0...0]$), mentre $\epsilon$ incarna un valore scalare infinitamente piccolo, tipicamente imposto nell'ordine di $\epsilon=10^{-7}$. Nonostante sia un approccio conveniente dal punto di vista dello sviluppo, il calcolo automatico tramite differenze finite introduce inevitabilmente due svantaggi fondamentali. Per cominciare, la stima del gradiente derivante da questa pratica computazionale potrebbe rivelarsi non sufficientemente accurata rispetto al vero gradiente analitico. In secondo luogo, dal punto di vista dell'ottimizzazione delle performance, i calcoli numerici si dimostrano enormemente più gravosi: è difatti richiesto che la funzione obiettivo venga valutata, ad ogni singola iterazione, un numero di volte almeno pari a $2D$, dove $D$ è la dimensione dello spazio delle variabili. Per ottenere un gradiente ancora più accurato, il numero di valutazioni della funzione salirebbe ulteriormente, intaccando i tempi di esecuzione.

##### Esempio Pratico

Per sedimentare i concetti analizzati, un esempio classico richiede l'impiego del solutore numerico per individuare il punto di minimo della seguente espressione matematica:

$$f(y, z) = (y+3)^2 + \sin(y) + (z+1)^2$$

Sapendo che la funzione presa in esame possiede una natura convessa, si ha la totale certezza formale circa la presenza di un solo e unico minimo globale. L'implementazione di questa problematica parte strutturando in Python la funzione per accogliere un array NumPy monodimensionale $x$ di forma (2,). All'interno dell'array, la prima componente farà le veci della variabile $y$, e per continuità la seconda componente assumerà il ruolo della variabile $z$. La funzione dovrà poi valutare le variabili e restituire formalmente il risultato di $f(y,z)$. Tramite l'apposita chiamata a `scipy.optimize.fmin_l_bfgs_b`, il sistema restituirà una tupla contenente tre variabili determinanti: il punto `x` che rappresenta la posizione stimata nello spazio del minimo trovato, la variabile `f` indicante il valore della funzione obiettivo toccato proprio in quel punto, e un terzo elemento `d` contenente un insieme di informazioni supplementari in merito alla risoluzione in corso.

La vera potenza dell'ottimizzazione affiora fornendo, anziché un'approssimazione numerica, un gradiente matematico esplicito. Derivando la funzione rispetto ad ambedue le coordinate spaziali si ottiene la seguente sintassi formale del gradiente:

$$\frac{\partial f(y,z)}{\partial y}=2(y+3)+\cos(y)$$

$$\frac{\partial f(y,z)}{\partial z}=2(z+1)$$

Risulta perciò sufficiente istruire la funzione affinché elabori assieme al valore di $f(x)$ anche l'array derivato per restituirlo in blocco all'ottimizzatore. Non fornendo più al calcolatore l'argomento `approx_grad`, l'algoritmo risolverà il problema istantaneamente grazie alle sue derivate esatte. Studiando nel dettaglio le metriche restituite all'interno della variabile aggiuntiva `d`, risalterà l'informazione `'funcalls'`, una dicitura che quantifica il numero esatto di volte in cui la funzione principale è stata interpellata dal sistema. Il numero vertiginosamente inferiore di `'funcalls'` registrato utilizzando il gradiente esplicito è l'evidenza schiacciante di quanto l'approssimazione numerica divenga economicamente meno sostenibile col moltiplicarsi delle dimensioni geometriche del dominio preso in esame.

In [2]:
def function(x):
    y=x[0]
    z=x[1]
    return (y+3)**2 + np.sin(y) + (z+1)**2
def grad(x):
    y=x[0]
    z=x[1]

    dy=2*(y+3)+np.cos(y)
    dz=2*(z+1)
    return np.array([dy,dz])

In [3]:
x0 = np.array([0.0,0.0])
x, f, d = scipy.optimize.fmin_l_bfgs_b(func=function, x0=x0, fprime=grad)

print("Punto di minimo (x):", x)
print("Valore della funzione (f):", f)
print("Dizionario delle informazioni (d):", d)

Punto di minimo (x): [-2.57747137 -0.99999927]
Valore della funzione (f): -0.3561430123647611
Dizionario delle informazioni (d): {'grad': array([-1.50318729e-06,  1.46120529e-06]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 7, 'nit': 6, 'warnflag': 0}


#### Regressione Logistica Binaria

La regressione logistica binaria è un modello di classificazione probabilistico utilizzato per prevedere l'appartenenza a una di due classi mutuamente esclusive. A differenza della regressione lineare, che produce valori continui, la regressione logistica trasforma l'output attraverso una funzione sigmoide, garantendo che le predizioni rimangono confinate nell'intervallo [0, 1], interpretabili come probabilità di appartenenza alla classe positiva.

Nel nostro caso specifico, implementeremo un classificatore binario per discriminare tra due specie di iris: iris virginica e iris versicolor, escludendo completamente iris setosa dal nostro dataset. Le etichette saranno codificate secondo il seguente schema:
- **1**: iris versicolor (classe positiva)
- **0**: iris virginica (classe negativa)

Questo approccio sfrutta l'ottimizzazione numerica attraverso l'algoritmo L-BFGS precedentemente descritto, minimizzando la funzione di cross-entropia binaria per determinare i coefficienti ottimali del modello.

In [20]:
%load_ext autoreload
%autoreload 2
from scripts.utils import *
from scripts.logisticreg import *
from scripts.bayes_error import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
D, L = load_iris_binary()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)

In [21]:
lambdas=[1e-3,1e-1,1.0]
pi_emp = np.sum(LTR == 1) / LTR.shape[0]
emp_prior_log_odds = np.log(pi_emp / (1 - pi_emp))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR,LTR,l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate = np.mean(LP != LVAL)
    llr=S_val-emp_prior_log_odds
    min_dcf=compute_min_dcf(llr,LVAL,0.5,1,1)

    predictions_bayes = get_bayes_decision(llr, 0.5, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, 0.5, 1, 1)

    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

Lambda: 0.001
J ottima: 1.100009e-01
Error Rate: 8.8%
DCF min: 0.0625
Actual DCF:0.11805555555555555

Lambda: 0.1
J ottima: 4.539407e-01
Error Rate: 11.8%
DCF min: 0.05555555555555555
Actual DCF:0.1111111111111111

Lambda: 1.0
J ottima: 6.316436e-01
Error Rate: 14.7%
DCF min: 0.1111111111111111
Actual DCF:0.16666666666666666



In [22]:
lambdas=[1e-3,1e-1,1.0]
pi_T = 0.8
target_prior_log_odds = np.log(pi_T / (1 - pi_T))

for l in lambdas:
    v_opt,J_min=weighted_trainLogReg(DTR,LTR,l,pi_T)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate = np.mean(LP != LVAL)
    llr=S_val-target_prior_log_odds
    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(llr, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)

    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

Lambda: 0.001
J ottima: 9.401035e-02
Error Rate: 11.8%
DCF min: 0.16666666666666666
Actual DCF:0.2222222222222222

Lambda: 0.1
J ottima: 3.606261e-01
Error Rate: 38.2%
DCF min: 0.05555555555555555
Actual DCF:0.7222222222222222

Lambda: 1.0
J ottima: 4.724715e-01
Error Rate: 52.9%
DCF min: 0.1111111111111111
Actual DCF:1.0

